# Ingestão da camada bronze

A ingestão dos arquivos será feita para o catálogo de dados do Databricks na camada bronze, ainda sem nenhuma transformação.

## Configurações iniciais

- Criação do schema bronze para receber os dados brutos

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dataquality_challenge.bronze")

- Navegar no diretório raw_data e criar as tabelas da bronze a partir dos arquivos CSV.

In [0]:
%python
import os

raw_path = "/Workspace/Users/bueno1982@gmail.com/data-quality-challenge/raw_data"

csv_files = [f for f in os.listdir(raw_path) if f.endswith(".csv")]

print(f"Arquivos CSV encontrados: {len(csv_files)}")
for f in csv_files:
    print(f"  - {f}")

for csv_file in csv_files:
    table_name = csv_file.replace(".csv", "")
    full_table_name = f"dataquality_challenge.bronze.{table_name}"
    file_path = f"{raw_path}/{csv_file}"

    print(f"\n--- Ingestão de {csv_file} -> {full_table_name} ---")

    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(file_path)
        )

        row_count = df.count()
        print(f"  Registros lidos: {row_count:,}")
        df.printSchema()

        (
            df.write.mode("overwrite")
            .saveAsTable(full_table_name)
        )
        print(f"  Tabela {full_table_name} criada com sucesso!")

    except Exception as e:
        print(f"  Erro ao criar tabela {full_table_name}: {e}")

- Validar se os registros foram inseridos corretamente nas tabelas da bronze

In [0]:
%python
tables_df = spark.sql("""
    SELECT table_name
    FROM dataquality_challenge.information_schema.tables
    WHERE table_schema = 'bronze'
      AND table_catalog = 'dataquality_challenge'    
    ORDER BY table_name        
""")

table_names = [row.table_name for row in tables_df.collect()]

if table_names:
    results = []
    for table_name in table_names:
        count = spark.sql(f"SELECT COUNT(*) AS total FROM dataquality_challenge.bronze.{table_name}").collect()[0][0]
        results.append((table_name, count))
    display(spark.createDataFrame(results, ["table_name", "total_registros"]))
else:
    print("Nenhuma tabela encontrada na camada bronze. Execute a célula de ingestão primeiro.")